## 📑 HAPS Redefaults
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AQA6H)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("example").getOrCreate()


from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
import boto3
import os

#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ"S3PREFIX" 

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `p-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

##### Schema of the table

In [0]:
table_name = 'HAPS_Redefaults'

schema = (
      "BusinessDate date,"
      "PortfolioName string,"
      "AgreementKey string," 
      "ExposureKey string,"
      "CDI string,"
      "AssetSegment string,"
      "ProductKey string,"
      "ProductDescription string,"
      "SubProductDescription string,"
      "ImplementationDate date,"
      "ImplementationType string,"
      "ImplementationTypeMapping string,"
      "IsActiveFlag boolean,"
      "ClosedDate date,"
      "ArrearsStatus string,"
      "ArrearsStatusENG string,"
      "ObjectiveStatusDate date,"
      "ObjectiveStatus string,"
      "GBVAtModificationDateEur decimal(19,2),"
      "GBVEur decimal(19,2),"
      "AllocatedCollateralValueREV decimal(19,2),"
      "DelinquentEur decimal(19,2),"
      "DelinquentExpenseEur decimal(19,2),"
      "NonPayerEventMonth string,"
      "`SourceBucket1+EventMonth` string,"
      "`SourceBucket2+EventMonth` string,"
      "`SourceBucket3+EventMonth` string,"
      "`SourceBucket4+EventMonth` string,"
      "`DueInstallmentBucket1+EventMonth` string,"
      "`DueInstallmentBucket2+EventMonth` string,"
      "`DueInstallmentBucket3+EventMonth` string,"
      "`DueInstallmentBucket4+EventMonth` string,"
      "ImplementedPaymentcategory string,"
      "InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"

delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date
#### type: ignore
)


###### Optimizer function

In [0]:
# %python
# # Helper function to create or replace table
# def create_or_replace_table(table_name, df, partition_col):
#     df.write.format("delta").mode("overwrite").partitionBy(partition_col).saveAsTable(table_name)
# def create_or_replace_table_no_partition(table_name, df):
#     df.write.format("delta").mode("overwrite").saveAsTable(table_name)
# def repair_and_analyze_table(table_name):
#     spark.sql(f"REPAIR TABLE {table_name} SYNC METADATA")
#     spark.sql(f"ANALYZE TABLE {table_name} COMPUTE STATISTICS")    


##### haps_portfolios

In [0]:
haps_portfolios = spark.sql(
f'''
SELECT DISTINCT A.PortfolioName
FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios  A
INNER JOIN (SELECT BusinessDate, MAX(IngestionTimeStamp) IngestionTimeStamp, PortfolioName 
            FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios
            GROUP BY  BusinessDate, PortfolioName) B
ON A.BusinessDate = B.BusinessDate AND A.PortfolioName = B.PortfolioName AND A.IngestionTimeStamp = B.IngestionTimeStamp
WHERE A.Report = 'HAPS_Redefaults'
AND A.IsActive = 1 
''' 
)
haps_portfolios.createOrReplaceTempView("haps_portfolios")

###### Portfolio

In [0]:
portfolio = spark.sql(
f'''
SELECT  p.PortfolioName, cast(P.AssignmentDate as date)
FROM `{env_prefix}-cdh`.primary.portfolio P
INNER JOIN haps_portfolios  HP ON HP.PortfolioName = P.PortfolioName
WHERE P.BusinessDate = {run_date!r}
'''
)
portfolio.createOrReplaceTempView("portfolio")  

###### Modifications

In [0]:
Modifications = spark.sql(
f'''
SELECT  DISTINCT NewExposureKey, RestructuringDate, RestructuringType FROM (
    SELECT 
    NewExposureKey,
    restructuringDate AS RestructuringDate,
    RestructuringType,  rank() over (partition by NewExposureKey order by restructuringDate desc, BusinessDate desc, RestructuringType desc) as Rank
FROM `{env_prefix}-cdh`.primary.restructuring r
INNER JOIN portfolio P ON P.PortfolioName = r.PortfolioName
WHERE RestructuringDate <= {run_date!r}
and BusinessDate <= {run_date!r}
-- and RestructuringType is not null
and r.SourceSystem='profits'
) where Rank = 1
'''
)
Modifications.createOrReplaceTempView("modifications")

##### Product

In [0]:
product = spark.sql(
f'''
SELECT ProductKey, ProductDescription FROM (
    SELECT DISTINCT ProductKey, ProductDescription, rank() over (partition by ProductKey order by BusinessDate desc) as Rank
    FROM `{env_prefix}-cdh`.primary.product
    WHERE BusinessDate <= {run_date!r}
) WHERE Rank = 1
'''
)
product.createOrReplaceTempView("product")

##### RefDates

In [0]:
LastWorkingDates = spark.sql(
f'''
SELECT CalendarYear,
CalendarMonth,
MAX(CalendarDate) AS LastWorkingDate
FROM  `{env_prefix}-cdh`.reference.ref_calendar
WHERE IsBusinessDay = 1 and CalendarDate <= {run_date!r}
GROUP BY CalendarYear, CalendarMonth
'''
)
LastWorkingDates.createOrReplaceTempView("LastWorkingDates")

##### dexposure

In [0]:
dexposure_eoms  = spark.sql(
f'''
SELECT  de.*
FROM `{env_prefix}-cdh`.derivative.dexposure de
INNER JOIN LastWorkingDates r 
on de.BusinessDate = r.LastWorkingDate
INNER JOIN Portfolio p
on p.PortfolioName = de.PortfolioName AND r.LastWorkingDate > p.AssignmentDate
WHERE de.IsActiveFlag = true
'''
)
dexposure_eoms.createOrReplaceTempView("dexposure_eoms")

In [0]:
dexposure_active  = spark.sql(
f'''
SELECT  
    A.BusinessDate
    , A.PortfolioName
    , REPLACE(A.ExposureKey,'GXR','') as ExposureKey
    , A.CDI
    , A.AssetSegment
    , A.AgreementKey
    , A.IsActiveFlag
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.OnBalance, A.BusinessDate) OnBalance
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.GBV, A.BusinessDate) GBV
    , A.ObjectiveStatus
    , A.ObjectiveStatusdate
    , e.ExposureStatus as ArrearsStatus
    , A.ExposureStatus as ArrearsStatusENG
    , A.DueAmount 
    , A.DueExpense
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.GBV, A.BusinessDate) as GBVEur
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.DueAmount, A.BusinessDate) as DueAmountEur
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.DueExpense, A.BusinessDate) as DueExpenseEur
    , B.GBVAtModificationDateEur
    , B.NewArrearsStatusENG
    , B.NewSourceBucket
    , A.ProductKey 
    , P.ProductDescription 
    , A.SubProductDescription
    , A.ClosedReason
    , A.ClosedDate
    , A.Currency
    , A.DaysPastDueWithMateriality
    , A.LastRestructuringType
FROM `{env_prefix}-cdh`.derivative.dexposure as A
INNER JOIN portfolio pf ON pf.PortfolioName = A.PortfolioName
LEFT JOIN Product p 
ON A.ProductKey = p.ProductKey -- and p.BusinessDate = A.BusinessDate
LEFT JOIN (select distinct BusinessDate, ExposureKey, ExposureStatus 
             from`{env_prefix}-cdh`.primary.exposure
             where  PartyRelationshipType = 'Primary Owner' 
                and SourceSystem = 'qca'
             ) as e 
on e.ExposureKey = RIGHT(A.ExposureKey, LEN(A.ExposureKey)-3) and e.BusinessDate = A.BusinessDate 
LEFT JOIN (
select ExposureKey, BusinessDate, `{env_prefix}-cdh`.derivative.converttoeur(e.Currency, e.GBV, e.BusinessDate) as GBVAtModificationDateEur, e.ExposureStatus as NewArrearsStatusENG, e.Bucket as NewSourceBucket
--, rank() over (partition by e.ExposureKey order by M.RestructuringDate desc) as Rank
    from `{env_prefix}-cdh`.derivative.dexposure e
    inner join Modifications M
    on e.ExposureKey = M.NewExposureKey 
    and e.BusinessDate = if(e.OpenDate > M.RestructuringDate, e.OpenDate, M.RestructuringDate)
    where e.SourceSystem = 'profits'
        and e.BusinessDate <= {run_date!r}
) B
on A.ExposureKey = B.ExposureKey
WHERE A.IsActiveFlag = true
AND A.BusinessDate = {run_date!r}
'''
)
dexposure_active.createOrReplaceTempView("dexposure_active")

In [0]:
dexposure_inactive = spark.sql(
f'''
SELECT 
    {run_date!r} as BusinessDate
    , A.PortfolioName
    , REPLACE(A.ExposureKey,'GXR','') as ExposureKey
    , A.CDI
    , A.AssetSegment
    , A.AgreementKey
    , false as IsActiveFlag
    , cast(0 as decimal(19,2)) OnBalance
    , cast(0 as decimal(19,2)) GBV
    , A.ObjectiveStatus 
    , A.ObjectiveStatusDate
    , e.ExposureStatus as ArrearsStatus
    , A.ExposureStatus as ArrearsStatusENG
    , cast(0 as decimal(19,2)) as DueAmount 
    , cast(0 as decimal(19,2)) as DueExpense
    , cast(0 as decimal(19,2)) GBVEur
    , cast(0 as decimal(19,2)) DueAmountEur    
    , cast(0 as decimal(19,2)) DueExpenseEur
    , B.GBVAtModificationDateEur    
    , B.NewArrearsStatusENG
    , B.NewSourceBucket
    , A.ProductKey 
    , P.ProductDescription 
    , A.SubProductDescription
    , A.ClosedReason
    , case when A.IsActiveFlag = true then coalesce(A.ClosedDate, A.BusinessDate) else c.NextBusinessDay end as ClosedDate
    , A.Currency
    , A.DaysPastDueWithMateriality
    , A.LastRestructuringType
    from (select e.*
      , rank() over (partition by e.ExposureKey order by e.BusinessDate desc) as Rank
      from `{env_prefix}-cdh`.derivative.dexposure e
      inner join portfolio pf ON pf.PortfolioName = e.PortfolioName
      where e.SourceSystem = 'profits'
      --  and e.IsActiveFlag = false
        and e.BusinessDate <= {run_date!r}
        ) AS A
LEFT JOIN Product p 
ON A.ProductKey = p.ProductKey -- and p.BusinessDate = A.BusinessDate
LEFT JOIN (select distinct BusinessDate, ExposureKey, ExposureStatus, Bucket  
             from`{env_prefix}-cdh`.primary.exposure
             where  PartyRelationshipType = 'Primary Owner' 
                and SourceSystem in ('qca', 'qc')
             ) e
on e.ExposureKey = RIGHT(A.ExposureKey, LEN(A.ExposureKey)-3) and e.BusinessDate = A.BusinessDate 
LEFT JOIN `{env_prefix}-cdh`.reference.ref_calendar c 
ON c.CalendarDate = A.BusinessDate
LEFT JOIN (
select ExposureKey, BusinessDate, `{env_prefix}-cdh`.derivative.converttoeur(e.Currency, e.GBV, e.BusinessDate) as GBVAtModificationDateEur, e.ExposureStatus as NewArrearsStatusENG, e.Bucket as NewSourceBucket
--, rank() over (partition by e.ExposureKey order by M.RestructuringDate desc) as Rank
    from `{env_prefix}-cdh`.derivative.dexposure e
    inner join Modifications M
    on e.ExposureKey = M.NewExposureKey 
    and e.BusinessDate = if(e.OpenDate > M.RestructuringDate, e.OpenDate, M.RestructuringDate)
    where e.SourceSystem = 'profits'
        and e.BusinessDate <= {run_date!r}
) B
on A.ExposureKey = B.ExposureKey
left anti join dexposure_active e_pf_act on e_pf_act.ExposureKey = A.ExposureKey and e_pf_act.BusinessDate = {run_date!r}
where A.Rank = 1
'''
)
dexposure_inactive.createOrReplaceTempView("dexposure_inactive")

In [0]:
dexposure = spark.sql(
f'''
select *
from dexposure_active

union all

select *
from dexposure_inactive

'''
)
dexposure.createOrReplaceTempView("dexposure")

In [0]:
from pyspark.sql.types import IntegerType
from pyspark.sql.types import StringType
from pyspark.sql.types import BooleanType
from pyspark.sql.functions import udf

def get_bucket_from_dpd( dpd : int ):
        
    if dpd is None:
        return None
    elif dpd <= 0:
        return 0
    elif 1 <= dpd <= 29:
        return 1
    elif 30 <= dpd <= 59:
        return 2
    elif 60 <= dpd <= 89:
        return 3
    elif 90 <= dpd <= 119:
        return 4
    elif 120 <= dpd <= 149:
        return 5
    elif 150 <= dpd <= 179:
        return 6
    elif 180 <= dpd <= 209:
        return 7
    elif 210 <= dpd <= 239:
        return 8
    elif 240 <= dpd <= 269:
        return 9
    elif 270 <= dpd <= 299:
        return 10
    elif 300 <= dpd <= 329:
        return 11
    elif 330 <= dpd <= 359:
        return 12
    elif 360 <= dpd <= 720:
        return 13
    elif dpd > 720:
        return 14


# Convert your Python function into a UDF
get_bucket_from_dpd_udf = udf(get_bucket_from_dpd, IntegerType())



def is_valid_tax_number(Input : str):
    if Input is None:
        return None
    elif not Input.isdigit():
        return None
    elif Input == '999999993':
        return None
    elif any(c in Input for c in ['+', '-', '.', ',', 'E', 'e']):
        return None 
    elif len(Input) == 8:
        Input = '0' + Input
    if len(Input) != 9:
        return None
    factor = 256
    check_digit = int(Input[8])
    value =0
    for i in range(0,8):
        digit = int(Input[i])
        value = value + digit * factor
        factor = factor // 2
    if value == 0:
        return False
    checksum = value % 11
    if checksum == 10:
        checksum=0
    if checksum == check_digit:
        return True
    else:
        return False

# Convert your Python function into a UDF
is_valid_tax_number_udf = udf(is_valid_tax_number, BooleanType())

dexposure_eoms = dexposure_eoms.withColumn("SourceBucket", get_bucket_from_dpd_udf(dexposure_eoms["DaysPastDueWithMateriality"]))
dexposure_eoms.createOrReplaceTempView("dexposure_eoms")

dexposure = dexposure.withColumn("SourceBucket", get_bucket_from_dpd_udf(dexposure["DaysPastDueWithMateriality"]))
dexposure.createOrReplaceTempView("dexposure")

##### ModificationsFull

In [0]:
ModificationsFull = spark.sql(f'''
SELECT DISTINCT
            NewExposureKey
        ,	RestructuringDate
        ,	case when A.LastRestructuringType not in ('', ' ') then A.LastRestructuringType ELSE  mod.RestructuringType end as RestructuringType
FROM Modifications mod 
INNER JOIN dexposure A ON A.ExposureKey = mod.NewExposureKey
WHERE A.BusinessDate = {run_date!r}	AND 
      A.ProductDescription NOT LIKE '%ΥΠΟ ΑΙΡΕΣΗ%' AND
      SUBSTRING(A.ProductKey, CHARINDEX(';',A.ProductKey)+1,LEN(A.ProductKey)) NOT IN ('6078', '6079', '4015', '4016', '7362', '7363') AND
      A.ProductKey NOT IN ('40568;40000', '40568;40001') 
     '''
)
ModificationsFull.createOrReplaceTempView("ModificationsFull")

##### Payments

In [0]:
Payments = spark.sql(
f'''
select p.* from `{env_prefix}-cdh`.primary.payment p 
INNER JOIN portfolio pf ON pf.PortfolioName = p.PortfolioName
where PaymentActualDate <= {run_date!r}
and isdownpaymentflag = 'true'  
and sourcesystem = 'profits'
'''
)
Payments.createOrReplaceTempView("Payments")

##### PaymentSchedule 

In [0]:
PaymentSchedule = spark.sql(
f'''
SELECT * FROM 
(
        SELECT ExposureKey, 
        InstallmentAmount, 
        cast(InstallmentDate as Date), 
        last_day(ps.InstallmentDate) EoMInstallmentDate, 
        rank() over (partition by ExposureKey order by ps.BusinessDate desc) as Rank
        FROM ( 
        SELECT ExposureKey, abs(InstallmentAmount) as InstallmentAmount, InstallmentDate, BusinessDate, ps.PortfolioName, InstallmentType
        FROM `{env_prefix}-cdh`.primary.paymentSchedule ps
        INNER JOIN portfolio pf ON pf.PortfolioName = ps.PortfolioName
        where ifnull(ImplementationFlag, 'true') = 'true'
        ) ps 
        INNER JOIN Portfolio p
        ON p.PortfolioName = ps.PortfolioName  
        WHERE 1=1 
        AND ps.InstallmentAmount <> 0 
        AND ps.InstallmentDate <= {run_date!r}
        AND ps.BusinessDate = {run_date!r}  
        AND ps.InstallmentType = 'Installment'  
        AND ps.InstallmentDate > last_day(p.AssignmentDate)  
) WHERE RANK = 1
UNION ALL 
SELECT * FROM 
(
        SELECT ExposureKey, PaymentAmount, PaymentActualDate, last_day(PaymentActualDate), rank() over (partition by ExposureKey order by p.BusinessDate desc) as Rank 
        FROM Payments p
        
) WHERE RANK = 1
'''
)
PaymentSchedule.createOrReplaceTempView("PaymentSchedule")

##### Installments

In [0]:
Installments = spark.sql(
    f'''
		SELECT 
					a.ExposureKey 
				,	a.EOMInstallmentDate
			   ,	SUM(a.InstallmentAmount) AS InstallmentAmount						
		FROM PaymentSchedule a
		INNER JOIN (
				SELECT 
							p.ExposureKey 
						,	p.EOMInstallmentDate
					    ,	MAX(p.InstallmentDate) AS InstallmentDate
				FROM PaymentSchedule p
				GROUP BY ExposureKey, p.EOMInstallmentDate) b	ON a.ExposureKey = b.ExposureKey AND a.InstallmentDate = b.InstallmentDate
		GROUP BY a.ExposureKey, a.EOMInstallmentDate
  '''
)
Installments.createOrReplaceTempView("Installments")

##### reexposure

In [0]:
reexposure = spark.sql(
    f'''
SELECT AgreementKey, SUM(AllocRevBase) as AllocRevBase
FROM 
( 
    select AgreementKey,  AllocRevBase
    from `{env_prefix}-cdh`.derivative.dlienanalysisrealestateexposure re
    inner join portfolio pf ON pf.PortfolioName = re.PortfolioName
    where BusinessDate = (SELECT DISTINCT PreviousBusinessDay from `{env_prefix}-cdh`.reference.ref_calendar  where calendarDate = {run_date!r}) 
)
group by AgreementKey
'''
)
reexposure.createOrReplaceTempView("reexposure")


##### danalyticsmetrics

In [0]:
danalyticsmetrics = spark.sql(
f'''
select da.AgreementKey, da.PrimaryPartyCDI, da.PayerFlagAccount, PayerFlagAgreement, PayerFlagEntity, BusinessDate 
from `{env_prefix}-cdh`.derivative.danalyticsmetrics da
inner join portfolio pf ON pf.PortfolioName = da.PortfolioName
where BusinessDate <= {run_date!r} and BusinessDate >= '2024-01-01'
'''
)
danalyticsmetrics.createOrReplaceTempView("danalyticsmetrics")

###### Load historical NonPayerEventMonth for dates up to 2023-12-29

In [0]:
NonPayerEventMonthHist = spark.sql(
f'''
select nph.*, 
CASE WHEN day(m.RestructuringDate) > day(a.BusinessDate) THEN  DATEDIFF(MONTH, m.RestructuringDate, a.BusinessDate) + 1 ELSE DATEDIFF(MONTH, m.RestructuringDate, a.BusinessDate) END as EventMonth
from `{env_prefix}-reportinputfiles`.repoint.haps_NonPayerEventMonthHistorical nph 
inner join ModificationsFull as M on M.NewExposureKey = nph.AccountNumber
INNER JOIN dexposure_eoms a ON M.NewExposureKey = A.ExposureKey AND M.RestructuringDate <= A.BusinessDate
where nph.NonPayerEventMonth > 0 
'''
)
NonPayerEventMonthHist.createOrReplaceTempView("NonPayerEventMonthHist")

##### EventInfo 

In [0]:
EventInfo = spark.sql(
f'''  
		SELECT 
					a.ExposureKey
				,	a.SourceBucket
				,	IF(I.InstallmentAmount IS NULL, NULL, 
						ROUND((`{env_prefix}-cdh`.derivative.converttoeur(A.Currency, a.DueAmount - a.DueExpense, A.BusinessDate)  / I.InstallmentAmount),0)
      				)	AS DueInstallmentBucket
          		,   replace(d.PayerFlagAccount, '_', ' ') as AccountPayerSegment2 
				, 	replace(d.PayerFlagAgreement, '_', ' ') as AccountPayerSegment
				, 	replace(d.PayerFlagEntity, '_', ' ') as AccountPayerSegment3 
				,	CASE WHEN day(m.RestructuringDate) > day(a.BusinessDate) THEN  DATEDIFF(MONTH, m.RestructuringDate, a.BusinessDate) + 1 ELSE DATEDIFF(MONTH, m.RestructuringDate, a.BusinessDate) END AS EventMonth
		FROM ModificationsFull AS M
		INNER JOIN dexposure_eoms a ON M.NewExposureKey = A.ExposureKey AND M.RestructuringDate <= A.BusinessDate
		LEFT JOIN Installments I ON I.ExposureKey = A.ExposureKey AND last_day(A.BusinessDate) = I.EOMInstallmentDate
		LEFT JOIN danalyticsmetrics d on d.AgreementKey = a.AgreementKey and d.PrimaryPartyCDI = A.CDI and  d.BusinessDate = A.BusinessDate

    '''
  )
EventInfo.createOrReplaceTempView("EventInfo")

##### EventInfoGrouped

In [0]:
EventInfoGrouped = spark.sql(
f'''
		SELECT 
					AI.ExposureKey
				,	MIN(IF(
                            CASE WHEN nph.AccountNumber IS NOT NULL THEN 'Non Payer' ELSE AI.AccountPayerSegment END = 'Non Payer', coalesce(nph.NonPayerEventMonth, AI.EventMonth), NULL ) )	AS NonPayerEventMonth
				,   MIN(IF(AI.SourceBucket >=1, AI.EventMonth, NULL ))							AS `SourceBucket1+EventMonth`
				,   MIN(IF(AI.SourceBucket >=2, AI.EventMonth, NULL ))							AS `SourceBucket2+EventMonth`
				,   MIN(IF(AI.SourceBucket >=3, AI.EventMonth, NULL ))							AS `SourceBucket3+EventMonth`
				,   MIN(IF(AI.SourceBucket >=4, AI.EventMonth, NULL ))							AS `SourceBucket4+EventMonth`
				,	MIN(IF(AI.DueInstallmentBucket >=1, AI.EventMonth, NULL ))					AS `DueInstallmentBucket1+EventMonth`
				,   MIN(IF(AI.DueInstallmentBucket >=2, AI.EventMonth, NULL ))					AS `DueInstallmentBucket2+EventMonth`
				,   MIN(IF(AI.DueInstallmentBucket >=3, AI.EventMonth, NULL ))					AS `DueInstallmentBucket3+EventMonth`
				,   MIN(IF(AI.DueInstallmentBucket >=4, AI.EventMonth, NULL ))					AS `DueInstallmentBucket4+EventMonth`
				,	MAX(AI.EventMonth) -1  AS MonthsActive
		FROM EventInfo AI
        LEFT JOIN NonPayerEventMonthHist nph on nph.AccountNumber = AI.ExposureKey and nph.EventMonth = AI.EventMonth
		WHERE 1=1
			  AND AI.EventMonth <> 0
		GROUP BY AI.ExposureKey
'''
)
EventInfoGrouped.createOrReplaceTempView("EventInfoGrouped")

##### Perimeter

In [0]:
redefaults_perimeter = spark.sql(
f'''
		SELECT 	
					a.PortfolioName
				,	a.AgreementKey
				,	M.NewExposureKey AS ExposureKey
				,	a.CDI
				,	a.AssetSegment
				,	a.ProductKey
				,	a.ProductDescription
				,	A.SubProductDescription
				,	M.RestructuringDate
				,	M.RestructuringType  
				,	map.ImplementationTypeProductKey AS ImplementationTypeMapping
				,	A.IsActiveFlag
				,	A.ClosedDate
				,	a.ArrearsStatus
				,	a.ArrearsStatusENG
				,	a.ObjectiveStatusDate 
				,	a.ObjectiveStatus
				,	a.GBVAtModificationDateEur
				,	a.GBVEur
				,	d.AllocRevBase AS AllocatedCollateralValueREV
				,	a.DueAmountEur AS DelinquentEur
				,	a.DueExpenseEur AS DelinquentExpenseEur
				,	AI.NonPayerEventMonth
				,	AI.`SourceBucket1+EventMonth`
				,	AI.`SourceBucket2+EventMonth`
				,	AI.`SourceBucket3+EventMonth`
				,	AI.`SourceBucket4+EventMonth`
				,	AI.`DueInstallmentBucket1+EventMonth`
				,	AI.`DueInstallmentBucket2+EventMonth`
				,	AI.`DueInstallmentBucket3+EventMonth`
				,	AI.`DueInstallmentBucket4+EventMonth`
		FROM ModificationsFull AS M
		LEFT JOIN EventInfoGrouped AI  ON AI.ExposureKey = M.NewExposureKey
		INNER JOIN dexposure a ON a.ExposureKey = M.NewExposureKey AND a.BusinessDate = {run_date!r}
		LEFT JOIN `{env_prefix}-reportinputfiles`.repoint.haps_implementedproductmapping AS map ON replace(map.ProductKey,'-',';') = a.ProductKey
		LEFT JOIN reexposure d on d.AgreementKey = A.AgreementKey 
		WHERE a.GBVAtModificationDateEur > 0
'''
)
redefaults_perimeter.createOrReplaceTempView("redefaults_perimeter")

##### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT 				CAST({run_date!r} AS DATE) AS BusinessDate
				,	CAST(PortfolioName AS STRING) AS PortfolioName
				,	CAST(AgreementKey AS STRING) AS AgreementKey
				,	CAST(ExposureKey AS STRING) AS ExposureKey
				,	CAST(CDI AS STRING) AS CDI
				,	CAST(AssetSegment AS STRING) AS AssetSegment
				,	CAST(ProductKey AS STRING) AS ProductKey
				,	CAST(ProductDescription AS STRING) AS ProductDescription
				,	CAST(SubProductDescription AS STRING) AS SubProductDescription
				,	RestructuringDate AS ImplementationDate
				,	CAST(RestructuringType AS STRING) AS ImplementationType
				,	CAST(ImplementationTypeMapping AS STRING) AS ImplementationTypeMapping
				,	CAST(IsActiveFlag AS BOOLEAN) AS IsActiveFlag
				,	CAST(ClosedDate AS DATE) AS ClosedDate
				,	CAST(ArrearsStatus AS STRING) AS ArrearsStatus
				,	CAST(ArrearsStatusENG AS STRING) AS ArrearsStatusENG
				,	CAST(ObjectiveStatusDate AS DATE) AS ObjectiveStatusDate
				,	CAST(ObjectiveStatus AS STRING) AS ObjectiveStatus
				,	CAST(GBVAtModificationDateEur AS DECIMAL(19,2)) GBVAtModificationDateEur
				,	CAST(GBVEur AS DECIMAL(19,2)) AS GBVEur
				,	CAST(AllocatedCollateralValueREV AS DECIMAL(19,2)) AS AllocatedCollateralValueREV
				,	CAST(DelinquentEur AS DECIMAL(19,2)) AS DelinquentEur
				,	CAST(DelinquentExpenseEur AS DECIMAL(19,2)) AS DelinquentExpenseEur
				,	CAST(NonPayerEventMonth	AS string)  NonPayerEventMonth				
				,	CAST(`SourceBucket1+EventMonth`	AS string) AS `SourceBucket1+EventMonth`			
				,	CAST(`SourceBucket2+EventMonth`	AS string) AS `SourceBucket2+EventMonth`			
				,	CAST(`SourceBucket3+EventMonth`	AS string) AS `SourceBucket3+EventMonth`			
				,	CAST(`SourceBucket4+EventMonth`	AS string) AS `SourceBucket4+EventMonth`			
				,	CAST(`DueInstallmentBucket1+EventMonth`	AS string) AS `DueInstallmentBucket1+EventMonth`	
				,	CAST(`DueInstallmentBucket2+EventMonth`	AS string) AS `DueInstallmentBucket2+EventMonth`	
				,	CAST(`DueInstallmentBucket3+EventMonth`	AS string) AS `DueInstallmentBucket3+EventMonth`	
				,	CAST(`DueInstallmentBucket4+EventMonth`	AS string) AS `DueInstallmentBucket4+EventMonth`
				,	CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
		FROM redefaults_perimeter p
  ''')
final_df.createOrReplaceTempView("final_df")

In [0]:
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
Redefaults = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
Redefaults.createOrReplaceTempView("Redefaults")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/HAPS/Redefaults/" # 
file_name = "Redefaults_" +run_date+ ".csv"
Redefaults.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}") # 
rename_and_cleanup_files(output_path, file_name)